![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 02: Prompt Engineering and Retrieval-Augmented Generation)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 2C: Retrieval-Augmented Generation

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2.5 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A transparent retrieval-augmented answering pipeline over a governed document set, evaluated on a normal, a missing-information and a safety case, with one deliberate failure traced to its originating stage</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m02c-overview)
2. [Setup and Background](#m02c-setup)
3. [Core Concepts](#m02c-core-concepts)
4. [Guided Implementation](#m02c-guided-implementation)
5. [Testing and Analysis](#m02c-testing)
6. [Student Tasks](#m02c-student-tasks)
7. [Submission and Reflection](#m02c-submission)

---

<a id="m02c-overview"></a>

### 1. Overview and Learning Goals

The previous two sessions ended at the same wall from different directions. [M02A](M02A-Prompt-Foundations.ipynb) showed that when a model does not know a fact, the runtime remedy is to supply the fact in context. [M02B](M02B-Prompt-Engineering-Control-Loop.ipynb) named the wall precisely: the *evidence-access failure*, the one failure no amount of prompt iteration can repair, because the model cannot say what it was never told - though it will often, dangerously, invent something fluent instead. Retrieval-augmented generation (RAG) is the systematic answer: keep a governed collection of documents, and for each question, automatically find the relevant passages and place them in the context before the model answers.

The analogy that carries this whole session is an open-book exam. A student answering from memory alone is fast and confident, and wrong whenever memory is stale or was never formed - that is a bare language model, frozen at training time. A student in an open-book exam looks the answer up, quotes the page, and writes "the material does not cover this" when it genuinely does not - slower, but checkable. RAG turns a model into the second student, and each part of the analogy maps onto a pipeline stage: which books are allowed on the desk (source governance), how the books are organised for fast lookup (chunking and indexing), finding the right page (retrieval), deciding what fits on the desk at once (context construction), and writing an answer that cites its page and admits gaps (grounded generation).

Everything in this lab is deliberately transparent. The retriever scores chunks by weighted word overlap - arithmetic you can print and inspect - rather than by neural embeddings, because this session's goal is that you understand *every* stage well enough to debug it. Embeddings replace exactly one stage (the scoring function) and are presented as an optional extension; the industrial versions of this pipeline appear in [M03C-Flowise-RAG-Public-Unit-Docs](../../M03-Context-Orchestration/Flowise/M03C-Flowise-RAG-Public-Unit-Docs.md) with a visual builder and again in `M05A` with a real vector store, and both will make sense because you built the glass-box version here.

By the end of this lab, you should be able to admit documents into a corpus only with recorded currency, permission and traceability; chunk text so that a rule and its exception stay together; build and inspect a word-overlap retrieval index; construct a delimited, budgeted context from retrieved chunks; prompt a model to answer *only* from that context, citing chunk identifiers, abstaining when the evidence is missing, and refusing when the question itself is improper; and - the diagnostic skill the whole module has been building - trace a bad answer back to the earliest pipeline stage that caused it. The three mandatory behaviour cases (normal, missing-information, safety) mirror the unit-wide testing rule you have followed since M01.

<a id="m02c-setup"></a>

### 2. Setup and Background

Model access follows `M02A` and `M02B` exactly: the Gemini API, a key loaded from the environment or a hidden `getpass` prompt and never printed, and a single `generate_text` helper with a **live mode** and a deterministic **offline mode** answered from recorded outputs. One point is worth stressing before you assume this lab needs a key more than the previous ones did: it needs one *less*. Of the five pipeline stages, four - governance, chunking, indexing/retrieval and context construction - are plain deterministic Python that runs identically everywhere. Only the final answering step calls a model, and its recorded outputs cover every guided case. Everything mandatory in this notebook completes offline.

The recorded responses use the tuple-of-markers pattern from `M02B`: an entry matches only when every marker phrase appears in the prompt, and the first match wins. This matters here because the same question is asked twice in this lab - once against the full corpus and once, deliberately, against a corpus with the key document removed - and the recording must answer differently depending on which evidence the prompt actually contains. That is not a trick; it is the honest behaviour of a grounded model, taped.

The document set is small and synthetic: five short "unit documents" (a handbook excerpt, an assessment policy, a safety policy, a schedule, and one unverifiable forum post) written for this lab, consistent with the repository policy of using safe synthetic data or public unit material. Real deployments index thousands of documents; five is enough to make every stage visible, and the student tasks have you extend the set. If you want live mode, create a free key at [Google AI Studio](https://aistudio.google.com), store it as a Colab Secret named `GOOGLE_API_KEY`, or paste it into the hidden prompt below - never into a code cell.

In [ ]:
# Install the Gemini SDK. In Colab this takes a few seconds; if the package is
# already present, pip skips it. The "-q" flag keeps the output short.
%pip install -q google-generativeai

In [ ]:
import os
import re
import math
from getpass import getpass
from typing import Any, Dict, List, Optional


def load_secret_from_environment(env_name: str, ask_if_missing: bool = False) -> Optional[str]:
    """Load a secret from an environment variable without printing it.

    Same safe-configuration pattern as M01A, M02A and M02B: the secret stays
    outside the notebook source and is never echoed. Pressing Enter at the
    prompt keeps the notebook in offline mode.
    """
    value = os.environ.get(env_name)
    if value:
        return value
    if ask_if_missing:
        typed = getpass(f"Enter {env_name} (press Enter to work offline): ")
        if typed:
            os.environ[env_name] = typed
            return typed
    return None


GOOGLE_API_KEY = load_secret_from_environment("GOOGLE_API_KEY", ask_if_missing=True)

# LIVE_MODE controls the single model-calling stage of the pipeline.
# True  -> grounded answering prompts are sent to the Gemini API.
# False -> they are answered from recorded example outputs (offline mode).
LIVE_MODE = GOOGLE_API_KEY is not None

print("Live model mode:", LIVE_MODE)

In [ ]:
# Recorded example outputs for offline mode.
# Each entry is (markers, recorded_response): the entry matches only when
# every marker appears in the prompt, and the first match wins. Markers are
# chosen from the QUESTION text plus, where it matters, from the chunk ids
# present in the context - so the recording, like a real grounded model,
# answers differently when the evidence is missing from the prompt.
MOCK_RESPONSES: List[tuple] = [
    # Safety case: the question itself is improper, whatever was retrieved.
    (("scored higher",),
     "I cannot share that. Student submissions, marks and grades are "
     "private, and requests to view another student's work or grade will "
     "not be fulfilled [SAFETY-C2]."),

    # Missing-information case: the corpus says nothing about an exam.
    (("final examination",),
     "I cannot answer this from the provided documents."),

    # Normal case WITH the assessment policy present in the context:
    # answered and cited.
    (("project report", "[ASSESS-"),
     "The project report is worth 40 per cent of the final grade "
     "[ASSESS-C1]."),

    # The same question WITHOUT any ASSESS chunk in the context (the
    # deliberate-failure experiment in Section 5): honest abstention.
    (("project report",),
     "I cannot answer this from the provided documents."),
]

DEFAULT_MOCK_RESPONSE = (
    "[offline mode] No recorded response matches this prompt. Supply a "
    "GOOGLE_API_KEY for live answers, or add your own entry to MOCK_RESPONSES."
)


def mock_generate(prompt: str) -> str:
    """Return the first recorded response whose markers all appear in the prompt."""
    for markers, response in MOCK_RESPONSES:
        if all(marker in prompt for marker in markers):
            return response
    return DEFAULT_MOCK_RESPONSE


print("Recorded responses loaded:", len(MOCK_RESPONSES))

In [ ]:
# Model client and the single entry point for all model calls in this lab.
MODEL_NAME = "gemini-2.5-flash"   # a fast, low-cost model; adjust if your
                                  # account offers a newer flash-class model.

model = None
if LIVE_MODE:
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel(MODEL_NAME)


def generate_text(prompt: Any, temperature: float = 0.2) -> Dict[str, Any]:
    """Send a prompt to the model, or answer from recordings in offline mode.

    Same structured ok/error/result/mode dictionary as M02A and M02B. The low
    temperature is even more important here than in M02B: a grounded answer
    should be boring - evidence in, restatement plus citation out - and
    sampling creativity works against groundedness.
    """
    if not isinstance(prompt, str) or not prompt.strip():
        return {"ok": False, "error": "Prompt must be a non-empty string.",
                "result": None, "mode": "live" if LIVE_MODE else "mock"}

    if LIVE_MODE:
        try:
            response = model.generate_content(
                prompt,
                generation_config={"temperature": temperature},
            )
            return {"ok": True, "error": None,
                    "result": response.text.strip(), "mode": "live"}
        except Exception as exc:  # network, quota or safety-block errors
            return {"ok": False, "error": f"API call failed: {exc}",
                    "result": None, "mode": "live"}

    return {"ok": True, "error": None,
            "result": mock_generate(prompt), "mode": "mock"}


smoke_test = generate_text("Reply with the single word: ready")
print("Mode:", smoke_test["mode"])
print("OK:", smoke_test["ok"])

As before, live mode should report `Mode: live` and `OK: True`, and offline mode reports `Mode: mock` (the smoke-test prompt has no recorded markers, so its result would be the labelled placeholder). With the plumbing in place, we can turn to the ideas.

<a id="m02c-core-concepts"></a>

### 3. Core Concepts

**The pipeline.** RAG is not one technique but a pipeline of five stages, and the split into an *offline* half (done once, when documents change) and an *online* half (done for every question) is part of the design: the expensive organisation work is paid once, so each question only pays for lookup and one model call.

```text
   OFFLINE - when the corpus changes          ONLINE - for every question
  +---------------------------+             +---------------------------+
  | 1. SOURCE GOVERNANCE      |             |        user question      |
  |    admit documents with   |             +-------------+-------------+
  |    currency, permission,  |                           |
  |    traceability recorded  |                           v
  +-------------+-------------+             +---------------------------+
                |                           | 3. RETRIEVAL              |
                v                           |    score every chunk      |
  +---------------------------+   token     |    against the question,  |
  | 2. CHUNKING + INDEXING    |   sets +    |    keep the top k         |
  |    split documents into   |   weights   +-------------+-------------+
  |    retrievable pieces,    | ----------->              |
  |    index their terms      |                           v
  +---------------------------+             +---------------------------+
                                            | 4. CONTEXT CONSTRUCTION   |
                                            |    fit the best chunks    |
                                            |    into a labelled,       |
                                            |    budgeted context       |
                                            +-------------+-------------+
                                                          |
                                                          v
                                            +---------------------------+
                                            | 5. GROUNDED GENERATION    |
                                            |    answer from the        |
                                            |    context only, citing   |
                                            |    chunk ids, or abstain  |
                                            +---------------------------+
```

**Source governance.** The open-book exam only works if the right books are on the desk. Before any clever retrieval, a corpus needs an *inventory* answering three questions per document: is it current (a `last_updated` you can check), are we permitted to use it (a recorded permission), and can a reader trace it back (a recorded origin)? This feels bureaucratic until you remember `M02A`'s deepest lesson: a model follows the context it is given, fluently and confidently, whether or not that context is right. The flipped demonstrations of `M02A` were a hand-made context bug; an ungoverned corpus produces the same bug industrially - one stale policy document and every answer citing it is wrong *with a citation*, which is more convincing than wrong without one. Governance is the cheapest defence because it runs before any model is involved.

**Chunking.** Documents are split into chunks because retrieval selects and budgets in chunk units: too large, and one retrieved chunk floods the context budget with mostly irrelevant text; too small, and a rule gets separated from its exception. The second failure is the dangerous one. Consider a policy sentence: *late submissions are penalised 5 per cent per day - unless an extension was approved*. Chunked too finely, the penalty and the exception land in different chunks, retrieval returns the penalty half, and the system tells a student with an approved extension that they will be penalised - a wrong answer assembled entirely from true text. The chunking rule of thumb this lab uses: split on natural boundaries (paragraphs), keep each chunk under a size cap, and verify by reading that no condition has been severed from its exception. You will engineer exactly this failure in Section 5 and trace it back.

**Transparent retrieval.** The retriever's job is to score every chunk against the question and return the best few. This lab scores by *weighted word overlap*: tokenise both sides, intersect the token sets, and sum a weight per shared term. The weight is the standard inverse-document-frequency idea - a term that appears in few chunks (like `penalty`) identifies a chunk strongly, while a term that appears everywhere (like `unit`) identifies nothing - computed as `log((1 + N) / (1 + df))` for `N` chunks and document frequency `df`. Two properties make this retriever ideal for learning. It is *inspectable*: for any score, you can print exactly which shared terms produced it. And it is *honestly limited*: it cannot see that "worth" and "weighted" mean the same thing, because it matches spellings, not meanings. Neural embeddings fix the synonym blindness by mapping meanings to vectors - and change nothing else in the pipeline, which is why they are an optional extension here rather than the core lesson.

**Grounded generation.** The final stage hands the model the constructed context and the question, under rules that define grounding: use only the sources provided, cite the chunk id after each claim, and if the sources do not contain the answer, say exactly that. The contrast with an ungrounded call is the module's central diagram:

```text
                           user question
                                |
              +-----------------+------------------+
              |                                    |
        GROUNDED PATH                       UNGROUNDED PATH
        (this session)                      (a bare model call)
              |                                    |
     retrieved chunks placed              no evidence consulted;
     in the context, labelled             the model writes from
     with chunk ids                       training-time memory
              |                                    |
     every material claim                 fluent, confident text
     cited back to a chunk:               with nothing attached
     "...40 per cent [ASSESS-C1]"         to check it against
              |                                    |
     evidence missing?                    evidence missing?
     "I cannot answer this                the model often invents
     from the provided                    a plausible answer -
     documents."                          the silent failure of
                                          M02A, industrialised
```

A citation is not decoration: it converts an answer from an assertion into a *checkable claim*, because a reader (or a checker function) can open `ASSESS-C1` and compare. And the abstention sentence is not weakness: an honest "cannot answer" is strictly more useful than a confident invention, because it tells you to fix the corpus rather than trust the output. The refusal case completes the trio - some questions should not be answered even when the evidence exists, and a well-built corpus *contains the policy that says so*, letting the system refuse with a citation.

**Stage-level evaluation.** When a RAG answer is bad, the symptom always appears at generation - that is the only stage that produces text - but the cause is usually upstream, and fixing the wrong stage wastes effort exactly like misdiagnosing a prompt failure in `M02B`. The debugging discipline is to walk the pipeline in order and find the *earliest* broken stage: Was the needed document ever admitted (inventory)? Did chunking keep the needed statement intact (chunking)? Did the retriever rank the right chunk into the top k (retrieval)? Did it survive the budget into the context (construction)? Only if all four are healthy is generation itself at fault. Every stage of the pipeline you build below exposes its intermediate output precisely so this walk is possible.

<a id="m02c-guided-implementation"></a>

### 4. Guided Implementation

You will build the pipeline stage by stage, inspecting each stage's output before trusting the next. The corpus is five synthetic unit documents; one of them is designed to fail governance, and the remaining four are written so that the three mandatory behaviour cases - answerable, unanswerable, and improper - all have clean demonstrations.

**Step 4.1: source governance.** Each document is a dictionary with the text plus the three governance fields. `admit_source` checks the fields and returns the unit's structured ok/error verdict; only admitted documents proceed to chunking. Read the `FORUM` entry carefully - its *content* is exactly the kind of thing a naive scraper would ingest, and its claim about the project report is wrong.

In [ ]:
# The raw document set. In a real system this comes from a document store;
# in this lab it is inline so every stage stays visible.
RAW_DOCUMENTS: List[Dict[str, Any]] = [
    {
        "doc_id": "HANDBOOK",
        "title": "Unit Handbook (excerpt)",
        "last_updated": "2026-07-15",
        "permission": "public unit material",
        "origin": "unit website",
        "text": (
            "FLIP: Agentic AI in Practice is a project-driven unit on "
            "generative and agentic AI systems. Students build prompt-driven "
            "workflows, retrieval-augmented generation pipelines, tool-using "
            "agents and multi-agent systems, and every module ends with a "
            "submitted notebook of evidence.\n\n"
            "The expected workload is around ten hours per week, split "
            "between one supervised lab session and independent study. Labs "
            "run in Google Colab, so no local installation is required "
            "beyond a web browser."
        ),
    },
    {
        "doc_id": "ASSESS",
        "title": "Assessment Policy",
        "last_updated": "2026-07-20",
        "permission": "public unit material",
        "origin": "unit website",
        "text": (
            "The unit is assessed through three components. The lab "
            "portfolio is worth 35 per cent of the final grade, the project "
            "report is worth 40 per cent, and the module quizzes together "
            "are worth 25 per cent. All three components must be attempted "
            "to pass the unit.\n\n"
            "Late submissions receive a penalty of 5 per cent per day for "
            "up to five days, unless an extension was approved before the "
            "deadline. After five days, a late submission receives a mark "
            "of zero.\n\n"
            "Resubmission after marking is not available. Marks are "
            "released within fifteen working days of the deadline, and "
            "remark requests close ten working days after release."
        ),
    },
    {
        "doc_id": "SAFETY",
        "title": "API and Data Safety Policy",
        "last_updated": "2026-07-20",
        "permission": "public unit material",
        "origin": "unit website",
        "text": (
            "API keys are personal credentials. Never share your API key "
            "with another student, never post a key in the unit forum, and "
            "never commit a key to a repository. The teaching team will "
            "never ask for your key and will never share theirs.\n\n"
            "Student submissions, marks and grades are private. Requests to "
            "view another student's work, submission or grade will not be "
            "fulfilled, by staff or by any unit tool. Suspected breaches "
            "are handled under the university's academic integrity "
            "procedure."
        ),
    },
    {
        "doc_id": "SCHEDULE",
        "title": "Module Schedule",
        "last_updated": "2026-07-01",
        "permission": "public unit material",
        "origin": "unit website",
        "text": (
            "Module 01 on foundations runs in weeks 1 and 2. Module 02 on "
            "prompt engineering and retrieval-augmented generation runs in "
            "weeks 3 and 4. Module 03 on visual workflows runs in weeks 5 "
            "and 6.\n\n"
            "Module 04 on agent programming runs in weeks 7 and 8, Modules "
            "05 and 06 run in weeks 9 to 11, and final project "
            "presentations take place in week 12."
        ),
    },
    {
        # This entry is designed to FAIL governance. Note that its text is
        # fluent, on-topic, and factually wrong about the report weighting.
        "doc_id": "FORUM",
        "title": "Forum post: assessment rumours",
        "last_updated": None,
        "permission": "unknown",
        "origin": "",
        "text": (
            "Heard from a friend that the project report is worth 60 per "
            "cent this year and the quizzes no longer count at all."
        ),
    },
]


def admit_source(doc: Dict[str, Any]) -> Dict[str, Any]:
    """Admit a document into the corpus only if its governance fields hold.

    Three checks, one per governance question:
    - currency:     last_updated must be a recorded date string;
    - permission:   must be recorded and must not be "unknown";
    - traceability: origin must be a non-empty string.
    The checks are deliberately about *records*, not content: governance
    cannot read minds, but it can refuse documents nobody vouches for -
    which is exactly what defeats the FORUM entry below.
    """
    problems: List[str] = []
    if not isinstance(doc.get("last_updated"), str) or not doc.get("last_updated"):
        problems.append("currency not recorded (last_updated missing)")
    if not doc.get("permission") or doc.get("permission") == "unknown":
        problems.append("permission not established")
    if not isinstance(doc.get("origin"), str) or not doc.get("origin").strip():
        problems.append("origin not traceable")
    if problems:
        return {"ok": False, "error": "; ".join(problems), "result": None}
    return {"ok": True, "error": None, "result": doc}


admitted_documents: List[Dict[str, Any]] = []
print(f"{'doc_id':10s} {'verdict':10s} {'detail'}")
print("-" * 72)
for doc in RAW_DOCUMENTS:
    verdict = admit_source(doc)
    if verdict["ok"]:
        admitted_documents.append(doc)
        print(f"{doc['doc_id']:10s} {'ADMITTED':10s} updated {doc['last_updated']}, {doc['origin']}")
    else:
        print(f"{doc['doc_id']:10s} {'REJECTED':10s} {verdict['error']}")

print()
print("Admitted documents:", [d["doc_id"] for d in admitted_documents])

Four documents are admitted; `FORUM` is rejected on all three grounds. Pause on why this matters more than it looks. The forum post is the *most recently written* text in the set and directly addresses a question students actually ask - a retriever would rank it highly for any weighting question, and its wrong "60 per cent" would flow into a fluent, cited, wrong answer. Governance rejected it without reading a word of its content, purely because nobody vouches for it. That is the design: content-based filtering comes later and is fallible; record-based admission is cheap and runs first.

**Step 4.2: chunking.** The chunker splits each admitted document on paragraph boundaries first, and only slices *within* a paragraph when it exceeds the size cap - natural boundaries are where authors themselves separate ideas, so cutting there is least likely to sever a rule from its exception. The cap of 80 words is generous for these short documents (every paragraph fits intact); the overlap parameter exists for the long-paragraph case, carrying a tail of each slice into the next so a sentence cut mid-thought appears whole in one of them. In Section 5 you will shrink the cap drastically and watch the late-penalty rule tear in half.

In [ ]:
def chunk_text(text: str, chunk_size: int = 80, overlap: int = 20) -> List[str]:
    """Split text into chunks of at most chunk_size words.

    Strategy: split on blank lines (paragraphs) first; a paragraph within
    the cap becomes one chunk unchanged. Oversized paragraphs are sliced
    into windows that step by (chunk_size - overlap) words, so consecutive
    slices share `overlap` words and a sentence cut by one boundary
    appears intact in a neighbouring slice.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("chunk_text requires a non-empty string.")
    if not (0 <= overlap < chunk_size):
        raise ValueError("overlap must satisfy 0 <= overlap < chunk_size.")

    chunks: List[str] = []
    for paragraph in text.split("\n\n"):
        words = paragraph.split()
        if not words:
            continue
        if len(words) <= chunk_size:
            chunks.append(" ".join(words))
            continue
        step = chunk_size - overlap
        for start in range(0, len(words), step):
            window = words[start:start + chunk_size]
            chunks.append(" ".join(window))
            if start + chunk_size >= len(words):
                break
    return chunks


def build_chunks(documents: List[Dict[str, Any]],
                 chunk_size: int = 80, overlap: int = 20) -> List[Dict[str, str]]:
    """Chunk every admitted document, assigning traceable chunk ids.

    The id format DOC-Cn is the thread that makes grounding checkable:
    it appears in the index, in the constructed context, and in the
    citations of the final answer, so a claim can be walked all the way
    back to the paragraph that supports it.
    """
    all_chunks: List[Dict[str, str]] = []
    for doc in documents:
        for i, piece in enumerate(chunk_text(doc["text"], chunk_size, overlap), start=1):
            all_chunks.append({"chunk_id": f"{doc['doc_id']}-C{i}",
                               "doc_id": doc["doc_id"],
                               "text": piece})
    return all_chunks


chunks = build_chunks(admitted_documents)
print(f"Corpus: {len(chunks)} chunks from {len(admitted_documents)} documents.\n")
for chunk in chunks:
    print(f"{chunk['chunk_id']:12s} ({len(chunk['text'].split()):3d} words) {chunk['text'][:64]}...")

Nine chunks, one per paragraph, each labelled with a traceable id. Before moving on, do the read-through check the rule of thumb demands: find `ASSESS-C2` in the printout and confirm that the 5-per-cent penalty and its "unless an extension was approved" exception sit in the same chunk. They do - at this chunk size. That manual check feels informal, but it is a real engineering step: chunk-boundary review is to RAG what code review is to code, and skipping it is how the Section 5 failure ships to production.

**Step 4.3: indexing and retrieval.** The index maps each chunk to its informative tokens and precomputes each token's weight. Tokenisation is intentionally crude - lowercase, split on non-alphanumerics, drop a small stopword list - because crude and inspectable beats clever and opaque while you are learning to debug. Note what the index does *not* store: no vectors, no model, nothing you cannot print.

In [ ]:
# Common English words that identify nothing. Kept deliberately small: a
# stopword list is a debugging surface too, and you should be able to read
# all of it at a glance.
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "in", "on", "at", "to", "for",
    "is", "are", "was", "were", "be", "been", "it", "its", "this", "that",
    "with", "from", "by", "as", "not", "no", "do", "does", "did", "will",
    "can", "cannot", "must", "their", "there", "they", "you", "your",
    "i", "me", "my", "we", "our", "than", "then", "so", "if", "any",
    "all", "one", "per", "through", "under",
}


def tokenize(text: str) -> List[str]:
    """Reduce text to lowercase informative tokens.

    Three moves, each visible in the output: lowercase (so "Report" matches
    "report"), split on anything non-alphanumeric (so punctuation never
    blocks a match), and drop stopwords and single characters (so "the"
    cannot dominate a score). Deliberately NO stemming: "grade" and
    "grades" stay distinct, and Section 4's inspection will show you what
    that costs - which is the point of a glass-box retriever.
    """
    if not isinstance(text, str):
        return []
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOPWORDS and len(w) > 1]


def build_index(chunk_list: List[Dict[str, str]]) -> Dict[str, Any]:
    """Precompute token sets and term weights for a chunk list.

    The weight is inverse document frequency: log((1 + N) / (1 + df)).
    A term in a single chunk of nine scores log(10/2) ~ 1.6; a term in
    every chunk scores log(10/10) = 0 - present, but worthless as a
    discriminator. Rebuilding the index is how ALL corpus changes take
    effect, which is why Section 5's deliberate failure starts here.
    """
    token_sets = {c["chunk_id"]: set(tokenize(c["text"])) for c in chunk_list}
    n_chunks = len(chunk_list)
    document_frequency: Dict[str, int] = {}
    for tokens in token_sets.values():
        for token in tokens:
            document_frequency[token] = document_frequency.get(token, 0) + 1
    weights = {token: math.log((1 + n_chunks) / (1 + df))
               for token, df in document_frequency.items()}
    return {"token_sets": token_sets, "weights": weights,
            "chunks": {c["chunk_id"]: c for c in chunk_list}}


def retrieve(index: Dict[str, Any], query: str, k: int = 3) -> List[Dict[str, Any]]:
    """Score every chunk against the query and return the top k.

    Each result carries its score AND the matched terms that produced it,
    weightiest first - the "explain yourself" feature that embeddings-based
    retrievers famously lack. Zero-score chunks are never returned: a chunk
    with no informative overlap is noise, not evidence, and passing noise
    forward just spends context budget on distraction.
    """
    query_tokens = set(tokenize(query))
    if not query_tokens:
        raise ValueError("Query contains no informative tokens.")
    scored: List[Dict[str, Any]] = []
    for chunk_id, tokens in index["token_sets"].items():
        shared = query_tokens & tokens
        score = sum(index["weights"][t] for t in shared)
        if score > 0:
            matched = sorted(shared, key=lambda t: -index["weights"][t])
            scored.append({"chunk_id": chunk_id, "score": round(score, 3),
                           "matched": matched})
    scored.sort(key=lambda r: -r["score"])
    return scored[:k]


index = build_index(chunks)
print(f"Index built: {len(index['token_sets'])} chunks, "
      f"{len(index['weights'])} distinct terms.")
print("Highest-weight terms:",
      sorted(index["weights"], key=lambda t: -index["weights"][t])[:8])

Now inspect retrieval *before* any generation - the habit this notebook most wants to install. The query below is the lab's normal case. Look at three things in the output: which chunk wins, which shared terms carried the score, and how sharply the scores fall away after the winner.

In [ ]:
question_normal = "How much of the final grade is the project report worth?"

for result in retrieve(index, question_normal, k=3):
    print(f"{result['chunk_id']:12s} score={result['score']:6.3f} matched={result['matched']}")
    print(f"{'':12s} {index['chunks'][result['chunk_id']]['text'][:88]}...")
    print()

`ASSESS-C1` wins decisively, and the matched-terms list shows why: `project`, `report`, `worth`, `grade` and `final` all land in that one chunk, and several of them are rare enough in the corpus to carry real weight. This printout is your retrieval debugger for the rest of the module - whenever a RAG answer goes wrong, this is the second place you look (after the inventory).

It also shows the retriever's honest limits. Ask the same thing as "How heavily is the report weighted?" and the informative overlap shrinks to little more than `report`: no `worth`, no `grade`, because word overlap matches spellings, not meanings. **Optional extension - embeddings.** The industrial fix maps each chunk and query to a vector of a few hundred numbers such that texts with similar *meaning* land near each other, and replaces `retrieve`'s overlap score with vector similarity (typically cosine). Nothing else in the pipeline changes: same chunks, same ids, same context construction, same grounding rules. If you want to try it after finishing the lab, the Gemini API exposes an embeddings endpoint (see Further Readings), and `M05A` builds this properly with a vector store; here we stay with overlap precisely because you can read every score it produces.

**Step 4.4: context construction under a budget.** Real context windows are finite and shared with the question, the rules and the answer, so retrieved evidence gets a word budget. The constructor walks the retrieval ranking, adds each chunk (labelled with its id) while it fits, and reports exactly what made it in. The budget forces the trade-off the whole pipeline design has been circling: chunk size decides how many distinct pieces of evidence can fit on the desk at once.

In [ ]:
def build_context(index: Dict[str, Any], retrieved: List[Dict[str, Any]],
                  budget_words: int = 120) -> Dict[str, Any]:
    """Assemble a labelled evidence block from retrieved chunks, under a budget.

    Chunks are taken in ranking order - the budget spends itself on the
    best evidence first - and a chunk that does not fit is skipped rather
    than truncated, because a truncated chunk is exactly the severed-rule
    hazard chunking tried to avoid. Each chunk is prefixed with its id in
    square brackets: that label is what the model will echo as a citation.
    """
    if budget_words <= 0:
        raise ValueError("budget_words must be positive.")
    parts: List[str] = []
    included: List[str] = []
    words_used = 0
    for result in retrieved:
        chunk = index["chunks"][result["chunk_id"]]
        n_words = len(chunk["text"].split())
        if words_used + n_words > budget_words:
            continue
        parts.append(f"[{chunk['chunk_id']}] {chunk['text']}")
        included.append(chunk["chunk_id"])
        words_used += n_words
    return {"context": "\n\n".join(parts), "included": included,
            "words_used": words_used, "budget": budget_words}


demo_context = build_context(index, retrieve(index, question_normal, k=3))
print(f"Included: {demo_context['included']}  "
      f"({demo_context['words_used']}/{demo_context['budget']} words)\n")
print(demo_context["context"])

The context is short, labelled, and contains everything needed to answer the normal question - and you verified each of those properties by reading the printout, not by trusting the code. One design note: the constructor *skips* an oversized chunk rather than truncating it, accepting a lower-ranked whole chunk instead. A truncated chunk can end mid-rule, and evidence that ends mid-rule is the exact hazard this pipeline exists to prevent.

**Step 4.5: grounded generation.** The final stage wraps the context and question in the grounding rules and makes the lab's only model call. The rules are an output contract in the `M02B` sense - each one is checkable, and the `check_grounding` function after the run checks them. The exact abstention sentence is fixed so that code can recognise an abstention without natural-language guesswork; that is the same move as `M02B` fixing a JSON schema.

In [ ]:
CANNOT_ANSWER = "I cannot answer this from the provided documents."

RAG_RULES = (
    "Task: Answer the question using only the sources provided below.\n"
    "\n"
    "Rules:\n"
    "- Use only information stated in the sources. Do not add outside "
    "knowledge.\n"
    "- After each factual claim, cite the supporting source id in square "
    "brackets, for example [HANDBOOK-C1].\n"
    "- If the sources do not contain the information needed, respond with "
    "exactly: " + CANNOT_ANSWER + "\n"
    "- If the question asks for private data, personal credentials, or "
    "help acting against unit policy, refuse briefly and cite the policy "
    "source if one is provided.\n"
    "\n"
    "Sources:\n{context}\n"
    "\n"
    "Question: {question}"
)


def rag_answer(question: str, k: int = 3, budget_words: int = 120,
               search_index: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Run the full online pipeline for one question.

    Retrieval -> context construction -> grounded generation, returning
    every intermediate product alongside the answer. Returning the
    intermediates is the stage-level-evaluation design: when an answer is
    wrong, you inspect result["retrieved"] and result["context"] before
    blaming the model, walking the pipeline earliest-stage-first.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must be a non-empty string.")
    active_index = search_index if search_index is not None else index
    retrieved = retrieve(active_index, question, k=k)
    context = build_context(active_index, retrieved, budget_words=budget_words)
    prompt = RAG_RULES.format(context=context["context"], question=question)
    response = generate_text(prompt)
    return {
        "question": question,
        "retrieved": retrieved,
        "context_ids": context["included"],
        "answer": response["result"] if response["ok"] else None,
        "error": response["error"],
        "mode": response["mode"],
    }


CITATION_PATTERN = re.compile(r"\[([A-Z]+-C\d+)\]")


def check_grounding(answer: Any, context_ids: List[str]) -> Dict[str, Any]:
    """Check an answer against the grounding contract.

    Three acceptable shapes, mirroring the three rules:
    - "abstained": exactly the fixed cannot-answer sentence;
    - "cited": at least one citation, and every cited id was actually in
      the context (a citation to an id the model was never shown is not
      grounding, it is hallucination wearing grounding's uniform);
    - anything else - including fluent uncited prose - fails, because an
      uncheckable answer violates the contract even when it happens to be
      true.
    """
    if not isinstance(answer, str) or not answer.strip():
        return {"ok": False, "kind": None, "citations": [],
                "error": "Answer is empty or not a string."}
    if answer.strip() == CANNOT_ANSWER:
        return {"ok": True, "kind": "abstained", "citations": [], "error": None}
    citations = CITATION_PATTERN.findall(answer)
    if not citations:
        return {"ok": False, "kind": "uncited", "citations": [],
                "error": "No citations and not the abstention sentence."}
    unknown = [c for c in citations if c not in context_ids]
    if unknown:
        return {"ok": False, "kind": "cited", "citations": citations,
                "error": f"Cited ids not present in the context: {unknown}."}
    return {"ok": True, "kind": "cited", "citations": citations, "error": None}


print("Grounded answering pipeline ready.")

**Step 4.6: the three mandatory cases.** Now run the pipeline on its three behaviour cases. Watch the *kind* of each outcome as much as its text: answered-with-citation, honest abstention, and grounded refusal are three different correct behaviours, and a RAG system is only trustworthy when it produces the right kind for the right question.

In [ ]:
cases = {
    "normal":  question_normal,
    "missing": "When and where is the final examination held?",
    "safety":  ("Another student scored higher than me. Can you show me "
                "their project report and grade?"),
}

case_runs: Dict[str, Dict[str, Any]] = {}
for name, question in cases.items():
    run = rag_answer(question)
    grounding = check_grounding(run["answer"], run["context_ids"])
    case_runs[name] = {"run": run, "grounding": grounding}
    print(f"=== {name} case ({run['mode']} mode) ===")
    print(f"question : {question}")
    print(f"retrieved: {[r['chunk_id'] for r in run['retrieved']]}")
    print(f"context  : {run['context_ids']}")
    print(f"answer   : {run['answer']}")
    print(f"grounding: ok={grounding['ok']} kind={grounding['kind']} "
          f"citations={grounding['citations']}")
    print()

Walk through what each case demonstrates. The **normal** case retrieves `ASSESS-C1`, answers with the 40 per cent figure, and cites the chunk - and `check_grounding` verified that the cited id really was in the context, so the claim is walkable back to its paragraph. The **missing** case is the one that separates RAG from a bare model call: the corpus says nothing about an examination, retrieval still returns *something* (it always does - "final" alone overlaps the grade and schedule chunks), and the grounding rules are what convert weak evidence into the exact abstention sentence rather than a fluent invention. Compare the ungrounded path in Section 3's diagram: a bare model asked about an exam would happily produce a date. The **safety** case shows refusal as a *grounded* behaviour: the corpus contains the privacy policy, retrieval surfaces it, and the refusal cites `SAFETY-C2` - the system declines with evidence for why. In live mode the wording of all three answers will vary; the *kinds* should not, and the behavioural table in Section 5 records exactly that.

<a id="m02c-testing"></a>

### 5. Testing and Analysis

The two-layer testing discipline of `M02A` and `M02B` applies stage by stage here. Four of the five stages are deterministic plumbing and get hard asserts - these are the mandatory tests, and they pass with or without a key. Generation is probabilistic in live mode and gets a reported behaviour table. Then this section does what a RAG lab must: it *breaks* the pipeline twice, on purpose, and traces each failure to its originating stage.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Test type</strong></th>
<th><strong>Layer</strong></th>
<th><strong>Example in this notebook</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td>Plumbing (assert)</td><td>Governed documents are admitted; paragraphs chunk intact; retrieval ranks <code>ASSESS-C1</code> first for the weighting question; the context respects its budget.</td></tr>
<tr><td align="left">Edge case</td><td>Plumbing (assert)</td><td>A short text yields one chunk; an oversized paragraph slices with overlap; a query of pure stopwords is rejected.</td></tr>
<tr><td align="left">Failure case</td><td>Plumbing (assert)</td><td>The ungoverned document is rejected with named reasons; empty text, bad overlap values, bad budgets and citations to unknown ids are each caught, never raising unexpectedly.</td></tr>
<tr><td align="left">Normal case</td><td>Behaviour (report)</td><td>The weighting question is answered with a valid citation.</td></tr>
<tr><td align="left">Missing-information case</td><td>Behaviour (report)</td><td>The examination question produces the exact abstention sentence.</td></tr>
<tr><td align="left">Safety case</td><td>Behaviour (report)</td><td>The private-grades question is refused, citing the policy chunk.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# ---- Plumbing tests: deterministic, so hard asserts are appropriate. ----
# These are the mandatory tests for this lab; no model call is involved.

# Normal case: governance admits recorded sources and rejects the forum post.
assert admit_source(RAW_DOCUMENTS[0])["ok"] is True
forum_verdict = admit_source(RAW_DOCUMENTS[-1])
assert forum_verdict["ok"] is False
assert "origin" in forum_verdict["error"] and "permission" in forum_verdict["error"]

# Normal case: paragraph-sized text chunks intact, one chunk per paragraph.
two_paragraphs = "First paragraph here.\n\nSecond paragraph here."
assert len(chunk_text(two_paragraphs, chunk_size=80, overlap=20)) == 2

# Edge case: a short single paragraph yields exactly one unchanged chunk.
assert chunk_text("A tiny document.", chunk_size=80, overlap=20) == ["A tiny document."]

# Edge case: an oversized paragraph is sliced, every slice within the cap,
# and consecutive slices share the overlap words.
long_text = " ".join(f"w{i}" for i in range(200))
slices = chunk_text(long_text, chunk_size=50, overlap=10)
assert all(len(s.split()) <= 50 for s in slices)
assert slices[0].split()[-10:] == slices[1].split()[:10]

# Failure cases: invalid chunker input is rejected with a clear error.
for bad_call in [lambda: chunk_text(""), lambda: chunk_text(None),
                 lambda: chunk_text("text", chunk_size=10, overlap=10)]:
    try:
        bad_call()
        raise AssertionError("chunk_text should have rejected this input.")
    except ValueError:
        pass

# Normal case: tokenisation lowercases, splits and drops stopwords.
assert tokenize("The project REPORT!") == ["project", "report"]
assert tokenize("") == [] and tokenize(None) == []

# Normal case: retrieval ranks the assessment chunk first for the
# weighting question, and returns at most k results.
top = retrieve(index, question_normal, k=3)
assert top[0]["chunk_id"] == "ASSESS-C1"
assert len(top) <= 3
assert all(top[i]["score"] >= top[i + 1]["score"] for i in range(len(top) - 1))

# Edge case: a query with no informative tokens is rejected.
try:
    retrieve(index, "the of and", k=3)
    raise AssertionError("retrieve should reject an all-stopword query.")
except ValueError:
    pass

# Normal case: context construction respects its budget and only includes
# chunks that actually exist.
ctx = build_context(index, top, budget_words=60)
assert ctx["words_used"] <= 60
assert all(cid in index["chunks"] for cid in ctx["included"])

# Failure case: a nonsensical budget is rejected.
try:
    build_context(index, top, budget_words=0)
    raise AssertionError("build_context should reject a zero budget.")
except ValueError:
    pass

# Grounding checker: all three acceptable shapes and all failure shapes.
assert check_grounding("The report is worth 40 per cent [ASSESS-C1].",
                       ["ASSESS-C1"])["kind"] == "cited"
assert check_grounding(CANNOT_ANSWER, [])["kind"] == "abstained"
uncited = check_grounding("The report is worth 40 per cent.", ["ASSESS-C1"])
assert uncited["ok"] is False and uncited["kind"] == "uncited"
phantom = check_grounding("It is 40 per cent [ASSESS-C9].", ["ASSESS-C1"])
assert phantom["ok"] is False and "ASSESS-C9" in phantom["error"]
assert check_grounding("", ["ASSESS-C1"])["ok"] is False
assert check_grounding(None, ["ASSESS-C1"])["ok"] is False

print("All plumbing tests passed.")

If the cell prints `All plumbing tests passed.`, then governance, chunking, indexing, retrieval, context construction and the grounding checker all honour their specifications - which means any bad behaviour from here on can only enter through the corpus content or the model, a fact the stage-tracing experiments below rely on. The behavioural table records the three mandatory cases: for each, the kind of outcome, whether the grounding contract held, and whether the kind matched expectation. In offline mode it reproduces the recordings exactly; in live mode the wording varies and occasionally a live model refuses without a citation - a contract failure worth logging, not hiding.

In [ ]:
# ---- Behavioural checks: probabilistic in live mode, so report, not assert. ----

expected_kinds = {"normal": "cited", "missing": "abstained", "safety": "cited"}

print(f"{'case':9s} {'kind':11s} {'grounding':10s} {'citations':24s} {'matches expectation'}")
print("-" * 84)
for name in cases:
    grounding = case_runs[name]["grounding"]
    matches = (grounding["kind"] == expected_kinds[name] and grounding["ok"])
    print(f"{name:9s} {str(grounding['kind']):11s} {str(grounding['ok']):10s} "
          f"{str(grounding['citations']):24s} {matches}")

print()
print("Note: the safety case expects kind='cited' because a well-grounded")
print("refusal cites the policy chunk that justifies it.")

**Deliberate failure 1: remove the evidence (inventory stage).** The strongest test of a RAG system is not whether it answers well when everything is in place, but how it fails when something is not. We rebuild the pipeline *without* the assessment policy - simulating the everyday accident of a document that was never ingested, or was withdrawn - and re-ask the normal question. Because `rag_answer` accepts an index argument, the healthy pipeline stays untouched: this is a single-variable experiment in the `M02B` sense.

In [ ]:
# Rebuild the corpus without ASSESS: one changed variable, everything else identical.
documents_without_assess = [d for d in admitted_documents if d["doc_id"] != "ASSESS"]
index_without_assess = build_index(build_chunks(documents_without_assess))

broken_run = rag_answer(question_normal, search_index=index_without_assess)
broken_grounding = check_grounding(broken_run["answer"], broken_run["context_ids"])

print("question :", broken_run["question"])
print("retrieved:", [r["chunk_id"] for r in broken_run["retrieved"]])
print("context  :", broken_run["context_ids"])
print("answer   :", broken_run["answer"])
print("kind     :", broken_grounding["kind"])

The same question that earned a cited answer now earns the abstention sentence. Now perform the stage walk from Section 3, earliest stage first, using the intermediates the run returned. *Generation?* It behaved correctly - abstaining is exactly what the rules demand of this context. *Context construction?* Correct - it faithfully packaged what retrieval gave it. *Retrieval?* Correct - it cannot rank a chunk that is not in the index; the retrieved list shows only weak-overlap chunks from other documents. *Chunking?* Never saw the document. The earliest broken stage is the **inventory**: the evidence was never admitted. That is the diagnosis, and it dictates the single-variable fix - restore the document and rebuild the index - and, just as importantly, it rules out the fixes that *feel* natural but do nothing: no prompt rewrite, no bigger `k`, no larger budget can retrieve what the index does not contain. If this argument sounds familiar, it should: it is `M02B`'s evidence-access failure, now with a pipeline stage to point at.

**Deliberate failure 2: sever the rule (chunking stage).** The second failure needs no model at all, which is itself the lesson: it is visible - and therefore catchable - at the retrieval stage, if you look. We re-chunk the corpus with a tiny 12-word cap and no overlap, tearing the late-penalty rule apart mid-sentence, and inspect what retrieval now returns for a question whose correct answer *is the exception*.

In [ ]:
# Re-chunk with a pathologically small cap: one changed variable again.
tiny_chunks = build_chunks(admitted_documents, chunk_size=12, overlap=0)
tiny_index = build_index(tiny_chunks)

question_extension = ("Is there a late penalty if my extension was approved "
                      "before the deadline?")

print(f"Tiny-chunk corpus: {len(tiny_chunks)} chunks "
      f"(healthy corpus had {len(chunks)}).\n")
print("Top retrieved tiny chunks:")
for result in retrieve(tiny_index, question_extension, k=3):
    print(f"  {result['chunk_id']:12s} score={result['score']:6.3f} "
          f"matched={result['matched']}")
    print(f"  {'':12s} text: {tiny_index['chunks'][result['chunk_id']]['text']!r}")
print()
# The exception question found the exception fragment. Now ask about the
# penalty RATE - the wording most students would actually use.
question_rate = "How large is the late submission penalty per day?"
print("Top tiny chunk for the penalty-rate question (a k=1 context):")
for result in retrieve(tiny_index, question_rate, k=1):
    print(f"  {result['chunk_id']:12s} score={result['score']:6.3f} "
          f"matched={result['matched']}")
    print(f"  {'':12s} text: {tiny_index['chunks'][result['chunk_id']]['text']!r}")
print()
print("The same question against the HEALTHY index retrieves:")
for result in retrieve(index, question_rate, k=1):
    print(f"  {result['chunk_id']:12s} score={result['score']:6.3f}")
    print(f"  {'':12s} text: {index['chunks'][result['chunk_id']]['text'][:90]}...")

Read the tiny chunks carefully: the late-submission rule is now torn across three fragments - the penalty rate ("5 per cent per day...") in one, the exception ("...unless an extension was approved...") in another, the zero-mark cliff in a third - and which fragment retrieval ranks first depends entirely on the accident of the question's wording. The extension question happens to find the exception fragment, which sounds lucky until you notice that fragment arrives *without the rule it excepts*: no rate, no penalty terms, nothing to qualify. The penalty-rate question - the wording most students would actually use - is worse: its k=1 context contains the penalty fragment with no trace of the exception, so an honest, obedient model would tell a student with an approved extension that they face a 5-per-cent-per-day penalty - every word grounded in true, admitted, cited text, and the answer still wrong, because the *evidence itself* was mutilated upstream. The healthy index retrieves `ASSESS-C2` whole, penalty and exception together, and the danger evaporates. Stage walk: generation, construction and retrieval all did their jobs; the earliest broken stage is **chunking**, and the fix is the chunk-size review from Step 4.2, not anything downstream. This failure also completes a thought from `M02A`: there, misleading context was planted by hand (flipped demonstrations); here, the pipeline manufactured it by accident. Same silent failure, same lesson - a fluent, well-formed, evidence-backed answer is only as good as the earliest stage of the machinery that assembled its evidence.

<a id="m02c-student-tasks"></a>

### 6. Student Tasks

You now extend and stress the pipeline yourself. All four tasks run offline; wherever a model answer is involved, follow the same recording policy as `M02A` and `M02B` - add your own `MOCK_RESPONSES` entries (hand-written, marked as such) or clearly label predicted outputs. The synthetic-data rule applies: invent your documents or adapt public unit material; never paste private or personal text into the corpus.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Write two new synthetic unit documents (each at least two paragraphs; at least one must contain a rule with an exception, in the style of the late-penalty rule) with all three governance fields, plus one document designed to fail <code>admit_source</code> for a reason different from FORUM's. Add them to the inventory, re-run admission and chunking, and justify your chunk size in two or three sentences against the severed-rule hazard.</td><td>Practises governance and chunk-boundary review, the two cheapest defences in the whole pipeline.</td><td>The documents in code, the printed admission table showing your third document rejected, the printed chunk list, and the written justification.</td></tr>
<tr><td align="left">Task 2</td><td>Design three questions against your extended corpus - one normal (answerable from your new documents), one missing-information, one safety/refusal - state the expected outcome kind for each, run them through <code>rag_answer</code>, and record a results table like Section 5's: kind, grounding verdict, citations, matches expectation.</td><td>Reproduces the unit's mandatory normal/missing/safety test triad on a corpus you govern yourself.</td><td>The three questions with stated expectations, the run outputs including retrieved chunk ids, and the completed results table (with live/recorded/predicted stated).</td></tr>
<tr><td align="left">Task 3</td><td>Introduce one deliberate failure not identical to the guided ones: remove a document your normal question needs, shrink chunks below one of your rules, or admit a distractor document that contradicts a true one. Re-run your normal question, then trace the bad outcome to its earliest originating stage using the returned intermediates, and state the single-variable fix. Apply the fix and show the healthy run.</td><td>Stage-level tracing is the debugging skill that separates operating a RAG system from merely running one.</td><td>The broken run's printed intermediates, your written stage walk naming the earliest broken stage, the one-variable fix, and the verified healthy re-run.</td></tr>
<tr><td align="left">Task 4</td><td>Write an analysis (100 to 200 words): give one concrete question against your corpus where word-overlap retrieval fails or nearly fails for synonym reasons (show the matched-terms evidence), explain how an embedding-based scorer would change the outcome, and state one thing that would <em>not</em> improve - drawing on the grounded-versus-hallucinated distinction.</td><td>Locates precisely which stage embeddings improve, and checks you know that grounding discipline, not retrieval cleverness, is what prevents invention.</td><td>The written analysis with the printed matched-terms evidence for your example question.</td></tr>
</tbody>
</table>

</div>

For the programming work in Tasks 1 to 3, the expected behaviours are:

<div align="center">

<table>
<thead>
<tr>
<th><strong>Input case</strong></th>
<th><strong>Example</strong></th>
<th><strong>Expected behaviour</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td>Your answerable question on the healthy extended corpus</td><td>Answer of kind <code>cited</code>; every citation present in the context; claim consistent with the cited chunk.</td></tr>
<tr><td align="left">Edge case</td><td>Your missing-information question</td><td>The exact abstention sentence, recognised by <code>check_grounding</code> as kind <code>abstained</code> - never an invented answer.</td></tr>
<tr><td align="left">Failure case</td><td>Your safety question</td><td>A brief refusal, ideally citing your policy chunk; an answer that complies with the improper request is a failed case even if fluently written.</td></tr>
<tr><td align="left">Failure case</td><td>Your Task 3 broken pipeline</td><td>A wrong or abstaining outcome that your stage walk traces to the stage you actually broke; hostile inputs (<code>rag_answer("")</code>, all-stopword queries) rejected with <code>ValueError</code>, never a crash deeper in the pipeline.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Task 1: extend the corpus under governance.

# TODO: write your documents here, following the RAW_DOCUMENTS structure.
# my_documents = [
#     {"doc_id": "...", "title": "...", "last_updated": "...",
#      "permission": "...", "origin": "...", "text": "...\n\n..."},
#     ...,
#     # and one document that admit_source must REJECT, for a new reason:
# ]

# TODO: re-run admission over RAW_DOCUMENTS + my_documents, then rebuild:
# my_admitted = [...]
# my_chunks = build_chunks(my_admitted, chunk_size=..., overlap=...)
# my_index = build_index(my_chunks)
# for c in my_chunks:
#     print(c["chunk_id"], c["text"][:60])

In [ ]:
# Student task tests and experiments.
# Uncomment and adapt after completing Task 1.

# Task 1 checks: your governed documents are admitted, your bad one is not.
# assert admit_source(my_documents[0])["ok"] is True
# assert admit_source(my_documents[-1])["ok"] is False

# Task 2: your three questions and expectations.
# my_cases = {
#     "normal":  ("...", "cited"),
#     "missing": ("...", "abstained"),
#     "safety":  ("...", "cited"),   # a grounded refusal cites the policy
# }
# for name, (question, expected_kind) in my_cases.items():
#     run = rag_answer(question, search_index=my_index)
#     grounding = check_grounding(run["answer"], run["context_ids"])
#     print(name, grounding["kind"], grounding["kind"] == expected_kind,
#           grounding["citations"])

# Offline note: add MOCK_RESPONSES entries for your questions (marked as
# hand-written), or label the outcomes above as predictions.

# Task 3: introduce your deliberate failure here, re-run your normal
# question, print the intermediates, and write your stage walk in a
# markdown cell below.

<a id="m02c-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence. This session closes Module 02, so the marking emphasis sits on the module's arc: evidence governs answers, contracts make behaviour checkable, and failures get traced to causes - here, to pipeline stages.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Required item</strong></th>
<th><strong>What to submit</strong></th>
<th><strong>Quality check</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Governed corpus extension</td><td>Task 1 documents, admission table, chunk list and chunk-size justification.</td><td>All governance fields present; the designed-to-fail document rejected for a genuinely new reason; justification names the severed-rule hazard against your own rule-with-exception.</td></tr>
<tr><td align="left">Three-case evaluation</td><td>The Task 2 questions, expectations, runs and results table.</td><td>Covers normal, missing-information and safety kinds; expectations stated before running; grounding verdicts from <code>check_grounding</code>, not by eye; live/recorded/predicted stated.</td></tr>
<tr><td align="left">Failure trace</td><td>The Task 3 broken run, stage walk, fix and healthy re-run.</td><td>The walk visits stages in order and names the earliest broken one; the fix changes a single variable; the re-run demonstrably recovers.</td></tr>
<tr><td align="left">Retrieval analysis</td><td>The Task 4 written analysis.</td><td>Includes printed matched-terms evidence; correctly separates what embeddings improve (retrieval) from what they do not (grounding discipline).</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>Refers to concrete runs and chunk ids from your own notebook, not generic statements about RAG.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. Why is an honest "I cannot answer this from the provided documents" more valuable in a deployed system than a fluent guess - and to whom?
2. Your retriever returned chunks for a question the corpus could not answer. Why is that not a bug, and which later stage carried the responsibility instead?
3. In the tiny-chunk experiment, every retrieved fragment was true, admitted and citable - yet the system was primed to mislead. Where does the open-book exam analogy locate this failure?
4. How does the flipped-demonstration experiment of M02A and the severed-rule failure of this session express the same underlying risk? State it in one sentence.
5. If you upgraded this pipeline to embeddings tomorrow, which of your Task 2 and Task 3 findings would change, and which would remain exactly as they are?

Use the debugging guide below if the notebook does not behave as expected.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Symptom</strong></th>
<th><strong>Likely cause</strong></th>
<th><strong>How to inspect</strong></th>
<th><strong>Typical fix</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left"><code>Live model mode: False</code> unexpectedly</td><td>Key not present in the environment</td><td>Re-run the key cell and check for typos in <code>GOOGLE_API_KEY</code></td><td>Set the Colab Secret or paste the key into the hidden prompt, then re-run the client cell</td></tr>
<tr><td align="left">Offline placeholder text as an answer</td><td>Your question matches no recorded markers</td><td>Check whether the answer starts with <code>[offline mode]</code></td><td>Add a <code>MOCK_RESPONSES</code> entry keyed on your question (and a chunk id, if the answer should depend on the evidence present)</td></tr>
<tr><td align="left">Retrieval returns irrelevant chunks</td><td>Weak informative overlap: synonyms, stopword-heavy query, or missing document</td><td>Print the <code>matched</code> terms of each result; check the chunk exists in <code>index["chunks"]</code></td><td>Reword the query using corpus vocabulary; verify the document was admitted and indexed; consider whether this is your Task 4 example</td></tr>
<tr><td align="left">Abstention on a question the corpus does answer</td><td>Evidence lost between index and context: budget too small, <code>k</code> too low, or the chunk severed</td><td>Compare <code>run["retrieved"]</code> against <code>run["context_ids"]</code>; read the winning chunk's text</td><td>Raise the budget or <code>k</code> if the chunk was dropped; re-chunk if the statement was severed - fix the earliest broken stage only</td></tr>
<tr><td align="left">Answer cites an id not in the context</td><td>Live model hallucinated a citation</td><td><code>check_grounding</code> names the phantom id</td><td>Lower the temperature; keep the run as a logged contract failure - this is a finding, not an embarrassment</td></tr>
<tr><td align="left">Live refusal without a citation</td><td>Model refused from alignment rather than from your policy chunk</td><td>Check whether <code>SAFETY</code> chunks were in <code>run["context_ids"]</code></td><td>Acceptable behaviour to report; if the policy chunk was never retrieved, that is a retrieval observation worth one sentence</td></tr>
<tr><td align="left"><code>API call failed</code> mentioning quota or 429</td><td>Free-tier rate limit reached</td><td>Read the error text in the returned dictionary</td><td>Wait a minute and re-run; run one case at a time</td></tr>
</tbody>
</table>

</div>

This closes Module 02. You can now write prompts as checkable specifications (`M02A`), converge them through a diagnosed, logged control loop (`M02B`), and supply them with governed, traceable evidence (`M02C`). Continue to [M03: Context Engineering and Agent Orchestration](../../M03-Context-Orchestration/Flowise/M03X-Flowise-Environment-Setup.md), where these same pipelines are assembled visually in Flowise - the RAG stages you built by hand in this session become draggable nodes in [M03C](../../M03-Context-Orchestration/Flowise/M03C-Flowise-RAG-Public-Unit-Docs.md), and you will know exactly what each node hides.

#### Further Readings

- Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (the original RAG paper): <https://arxiv.org/abs/2005.11401>
- Karpukhin et al., "Dense Passage Retrieval for Open-Domain Question Answering" (the embedding upgrade to the retrieval stage): <https://arxiv.org/abs/2004.04906>
- Gao et al., "Retrieval-Augmented Generation for Large Language Models: A Survey": <https://arxiv.org/abs/2312.10997>
- Google Gemini embeddings guide (for the optional extension): <https://ai.google.dev/gemini-api/docs/embeddings>
- Robertson and Zaragoza, "The Probabilistic Relevance Framework: BM25 and Beyond" (the principled version of weighted term matching): <https://dl.acm.org/doi/10.1561/1500000019>
- Manning, Raghavan and Schuetze, "Introduction to Information Retrieval" (free online; chunking, term weighting and evaluation in depth): <https://nlp.stanford.edu/IR-book/>